# Error Analysis

Open predictions and group common failure modes.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT / 'src'))

from collections import Counter
from structura.dataset import read_jsonl
from structura.validators import hallucinated_product_ids, validate_output

path = ROOT / 'data/structura/predictions/rules_baseline_predictions.jsonl'
records = read_jsonl(path)
errors = []
for record in records:
    validation = validate_output(record['prediction'])
    if not validation.schema_valid:
        errors.append('schema_error')
        continue
    hallucinated = hallucinated_product_ids(validation.parsed, record['input']['retrieved_context'])
    if hallucinated:
        errors.append('hallucinated_product_id')
    elif validation.parsed['intent'] != record['target']['intent']:
        errors.append('wrong_intent')
Counter(errors)